# Acceptance: what summer4 sets up for you

**This is a sign-off page, not a tutorial.** It exercises roadmap step 49 (`plans/runtime-defaults.plan.md`;
composability findings `CX43` and `CX44`). When you `import summer4`, three things happen without any
set-up: JAX is shown one CPU device per core, `run()` uses an accurate adaptive solver (Tsit5) instead of
fixed-step Euler, and MCMC chains run in parallel when there are enough devices. Nothing here asserts;
the automated checks live in `tests/test_runtime_devices.py`, `tests/test_default_solver.py` and
`tests/test_chain_method.py`. The guide page `docs/guide/10-runtime.md` explains each default in words.

Run every cell top to bottom (under a minute). Sign off only if every **What to check** box holds.

| | You are checking that… |
|---|---|
| A | importing summer4 gives JAX one CPU device per core, and `SUMMER4_HOST_DEVICES=0` turns that off |
| B | on a predator-prey model, Euler spirals away from the true cycle and the default stays on it |
| C | the default is much faster and far more accurate than Euler (timings measured on the Mac mini, 12 cores) |
| D | four MCMC chains run faster in parallel than vectorized (same measurement) |
| E | `set_default_solver("euler")` and `SUMMER4_SOLVER=euler` bring back the old fixed-step numbers |

In [ ]:
import json
import os
import subprocess
import sys
from dataclasses import asdict
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.io as pio

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

import summer4
from summer4 import (
    Compartments,
    EntryFlow,
    ExitFlow,
    FlowModel,
    FlowRef,
    Param,
    Property,
    PropertyMap,
    Reduce,
    SavePlan,
    SaveRequest,
)
from summer4.solvers import set_default_solver

## A. One CPU device per core

`summer4.runtime_info()` says what summer4 set up in this process: whether 64-bit numbers are on,
how many CPU devices it asked JAX for and why, and which solver `run()` uses when you name none.
The table shows it for this notebook's process.

The bar chart compares what a fresh Python process sees with the default and with
`SUMMER4_HOST_DEVICES=0` (the feature switched off). Each bar is a separate child process that
imports summer4 and asks JAX how many devices it has. `JAX_PLATFORMS=cpu` keeps the count to the CPU
on any machine.

In [ ]:
pd.Series(asdict(summer4.runtime_info()), name="this process").to_frame()

In [ ]:
COUNT = "import summer4, jax; print(jax.device_count())"


def device_count(**env_changes: str) -> int:
    env = {k: v for k, v in os.environ.items() if k not in ("SUMMER4_HOST_DEVICES", "XLA_FLAGS")}
    env["JAX_PLATFORMS"] = "cpu"
    env.update(env_changes)
    out = subprocess.run(
        [sys.executable, "-c", COUNT], env=env, capture_output=True, text=True, check=True
    )
    return int(out.stdout.split()[-1])


counts = pd.Series(
    {
        "default (SUMMER4_HOST_DEVICES unset)": device_count(),
        "SUMMER4_HOST_DEVICES=0": device_count(SUMMER4_HOST_DEVICES="0"),
    },
    name="devices",
)
counts.plot.bar(
    title=f"CPU devices JAX sees after import summer4 ({os.cpu_count()} cores on this computer)",
    labels={"index": "how the process was started", "value": "jax.device_count()"},
)

> **What to check**
>
> - The default bar equals the number of cores named in the chart title; the `SUMMER4_HOST_DEVICES=0` bar is 1.
> - The table's `host_devices_source` is `default` (or the variable or flag you started this notebook with), and `x64` is `True`.

## B. The default solver keeps a cycle closed

The predator-prey model of guide page 09 cycles forever: the phase plane (predators against prey) is a closed loop.
A solver that is not accurate enough turns the loop into a spiral. The cell builds the model, then runs it three ways:

- **Euler** with `dt=0.1`, as the documentation used to;
- **the default**, with no `solver=` argument;
- **a reference**: `dopri5` with tolerance `1e-10`, far tighter than anything else here.

The figure draws all three in the phase plane.

In [ ]:
species = Property("species", ("prey", "predator"))
pmap = PropertyMap.from_property(species)
lv = FlowModel(pmap)
lv.add_flow(EntryFlow("prey_birth", species["prey"], Param("alpha") * Reduce(where=species["prey"])))
lv.add_flow(ExitFlow("prey_death", species["prey"], Param("mu")))
lv.add_flow(
    ExitFlow("predation", species["prey"], Param("a") * Reduce(where=species["predator"]))
)
lv.add_flow(
    EntryFlow("predator_birth", species["predator"], Param("e") * FlowRef("predation").sum_by())
)
lv.add_flow(ExitFlow("predator_death", species["predator"], Param("d")))
lv.set_initial_population({species["prey"]: 40.0, species["predator"]: 15.0})
lv_compiled = lv.compile()

lv_params = {"alpha": 0.6, "mu": 0.1, "a": 0.02, "e": 0.5, "d": 0.25}
lv_plan = SavePlan(
    requests={"compartments": SaveRequest(Compartments())}, ts=np.linspace(0.0, 100.0, 1001)
)


def lv_frame(**solver_kwargs) -> pd.DataFrame:
    result = lv_compiled.run(lv_params, t0=0.0, t1=100.0, dt=0.1, save=lv_plan, **solver_kwargs)
    frame = result["compartments"].to_pandas()
    frame.columns = ["prey", "predator"]
    return frame


reference = lv_frame(solver="dopri5", rtol=1e-10, atol=1e-10)
default = lv_frame()
euler = lv_frame(solver="euler")

fig = reference.plot(
    x="prey",
    y="predator",
    title="Predator-prey phase plane: reference, default solver and Euler",
    labels={"prey": "prey (animals)", "predator": "predators (animals)"},
)
fig.data[0].update(name="reference (dopri5, tol 1e-10)", line={"width": 6, "color": "lightgrey"})
fig.add_scatter(x=default["prey"], y=default["predator"], mode="lines", name="default (Tsit5)")
fig.add_scatter(x=euler["prey"], y=euler["predator"], mode="lines", name="Euler, dt=0.1")
fig.update_layout(showlegend=True)
fig.show()

> **What to check**
>
> - The Euler line spirals outwards: its loops get bigger each cycle, and it leaves the grey reference loop.
> - The default line lies on top of the thick grey reference loop; you should not be able to see them apart.

## C. Faster and more accurate, measured

The numbers in this section and the next were measured once, on the Mac mini (12 cores), by
`benchmarks/runtime_defaults.py`, and stored in `benchmarks/results/runtime_defaults.json`. They are read
from that file, not re-timed here, so they do not depend on the computer running this notebook.

For three models (an SIR epidemic, the predator-prey model, and a tuberculosis-scale model) the first chart
is the time for one `run()` after compiling, in milliseconds. The second is the largest relative error
against a tight `dopri5` reference. Both axes are logarithmic: each gridline is ten times the one before.

In [ ]:
RESULTS = Path("..") / ".." / "benchmarks" / "results" / "runtime_defaults.json"
if not RESULTS.exists():
    RESULTS = Path("benchmarks") / "results" / "runtime_defaults.json"
bench = json.loads(RESULTS.read_text())

rows = [
    {
        "model": model,
        "solver": solver,
        "ms per run": r["median_s"] * 1e3,
        "max relative error": max(r["max_rel_error"], 1e-12),
    }
    for model, by_solver in bench["solver"]["models"].items()
    for solver, r in by_solver.items()
]
solvers = pd.DataFrame(rows)
solvers["solver"] = solvers["solver"].replace(
    {
        "euler": "Euler (documented dt)",
        "default": "default (Tsit5, 1e-6)",
        "tsit5_loose": "Tsit5, loose (1e-4)",
    }
)
time_table = solvers.pivot(index="model", columns="solver", values="ms per run")
error_table = solvers.pivot(index="model", columns="solver", values="max relative error")
time_table.plot.bar(
    barmode="group",
    log_y=True,
    title="Time for one run (measured on the mini, 12 cores)",
    labels={"value": "milliseconds per run (log scale)", "model": "model", "solver": ""},
).show()
error_table.plot.bar(
    barmode="group",
    log_y=True,
    title="Largest relative error against a tight reference (values at 1e-12 are 'zero')",
    labels={"value": "max relative error (log scale)", "model": "model", "solver": ""},
).show()

> **What to check**
>
> - In the first chart the default bar is far below the Euler bar for every model.
> - In the second chart the default bar is far below the Euler bar too, for every model: the default is both faster and more accurate.
> - The loose Tsit5 bars are a little faster than the default and less accurate, which is the trade the default's tolerances were chosen for.

## D. Parallel chains

The same file holds a timing of the guide-08 calibration: 4 chains of 500 warm-up and 500 sampling
steps, with the mini's 12 devices. "First call" includes compiling; "second call" is the same
fit again, so it is what you pay for every refit. Both methods reached the same R-hat and effective sample
size, shown in the table under the chart.

In [ ]:
chains = pd.DataFrame(bench["chains"]["methods"]).T
chains[["first_call_s", "second_call_s"]].rename(
    columns={"first_call_s": "first call (compiles)", "second_call_s": "second call"}
).plot.bar(
    barmode="group",
    title="Four chains, 500 warm-up + 500 samples (measured on the mini, 12 cores)",
    labels={"index": "chain method", "value": "seconds", "variable": ""},
).show()
chains[["max_r_hat", "min_ess_bulk"]].round(3)

> **What to check**
>
> - Both bars of `parallel` are lower than the matching bars of `vectorized`.
> - The table below the chart shows the same `max_r_hat` and `min_ess_bulk` for both: parallel changes the speed, not the answer.

## E. Getting the old numbers back

Some work needs fixed-step numbers: a test that compares against a hand calculation, or a comparison with an old result.
A one-compartment decay model has an exact answer, 100 e^(-5) = 0.674 at time 5. Euler with a step of 0.5 is
inaccurate on it; the default is not. The table runs it with:

- no setting (the default);
- `set_default_solver("euler")` for the whole process, then `set_default_solver(None)` to restore the default;
- an explicit `solver="euler"` on the call;
- a fresh process started with `SUMMER4_SOLVER=euler`.

The second column asks `runtime_info()` which solver is the default at that point.

In [ ]:
DECAY = '''
import numpy as np
from summer4 import Compartments, ExitFlow, FlowModel, Param, Property, PropertyMap, SavePlan, SaveRequest

state = Property("state", ("A",))
pm = PropertyMap.from_property(state)
decay = FlowModel(pm)
decay.add_flow(ExitFlow("decay", state["A"], Param("k")))
decay.set_initial_population({state["A"]: 100.0})
decay_compiled = decay.compile()
plan = SavePlan(requests={"c": SaveRequest(Compartments())}, ts=np.array([0.0, 5.0]))


def value_at_5(**kwargs) -> float:
    result = decay_compiled.run({"k": 1.0}, t0=0.0, t1=5.0, dt=0.5, save=plan, **kwargs)
    return float(np.asarray(result["c"].to_pandas().iloc[-1, 0]))
'''
exec(DECAY)

exact = 100.0 * float(np.exp(-5.0))
table = {"exact": (exact, "-")}
table["no setting"] = (value_at_5(), summer4.runtime_info().default_solver)
set_default_solver("euler")
table["set_default_solver('euler')"] = (value_at_5(), summer4.runtime_info().default_solver)
set_default_solver(None)
table["solver='euler' on the call"] = (value_at_5(solver="euler"), "(explicit)")
table["after set_default_solver(None)"] = (value_at_5(), summer4.runtime_info().default_solver)

child_code = DECAY + "\nimport summer4\nprint(value_at_5(), '|', summer4.runtime_info().default_solver)"
child = subprocess.run(
    [sys.executable, "-c", child_code],
    env={**os.environ, "SUMMER4_SOLVER": "euler", "JAX_PLATFORMS": "cpu"},
    capture_output=True,
    text=True,
    check=True,
).stdout.strip().split("\n")[-1].split(" | ")
table["fresh process, SUMMER4_SOLVER=euler"] = (float(child[0]), child[1])

pd.DataFrame(table, index=["value at t=5", "default_solver says"]).T

> **What to check**
>
> - `exact` is 0.6738. "no setting" and "after set_default_solver(None)" agree with it to about six digits.
> - `set_default_solver('euler')`, `solver='euler' on the call` and the fresh `SUMMER4_SOLVER=euler` process all give the same Euler value, 0.0977 (far from 0.6738), and the second column names Euler where it was set.